In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
import os



r_seed = 42

WORKDIR = 'C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro/'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

C:\Users\cristiano.nascimento\AppData\Local\Temp\ipykernel_20280\731389767.py:15: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch
c:\Users\cristiano.nascimento\git\LSTM_giroscopio_acelerometro\.venv\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
date = datetime.now()
date = date.strftime("%d/%m/%Y_%H:%M")
arq = datetime.now()
arq = arq.strftime("%d_%m_%Y_%H_%M")

In [3]:
def combine_csv_data(directory):
    # Specify the directory containing the TXT files

    # Create an empty list to store the DataFrames
    dfs = []

    # Iterate through the TXT files in the directory
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            # Read the CSV data from the TXT file, assuming no header row
            df = pd.read_csv(os.path.join(directory, filename), header=None)

            # Append the DataFrame to the list
            dfs.append(df)

    # Concatenate the DataFrames into a single DataFrame
    combined_df = pd.concat(dfs, ignore_index=True)

    # Print the combined DataFrame
    return combined_df

In [4]:
def equalize_dataframe_sizes(df1, df2):
    """Equalizes the sizes of two DataFrames by dropping rows from the larger one.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        tuple: A tuple containing the equalized DataFrames (df1, df2).

    Raises:
        ValueError: If both DataFrames have the same size.
    """

    if df1.shape[0] == df2.shape[0]:
        raise ValueError("Both DataFrames already have the same size.")

    larger_df, smaller_df = (df1, df2) if df1.shape[0] > df2.shape[0] else (df2, df1)

    # Drop rows from the larger DataFrame to match the smaller one
    larger_df = larger_df.iloc[: smaller_df.shape[0], :]

    return larger_df, smaller_df

# Example usage
#df1 = pd.DataFrame({'A': [1, 2, 3, 4, 5], 'B': ['a', 'b', 'c', 'd', 'e']})
#df2 = pd.DataFrame({'A': [10, 20, 30], 'B': ['x', 'y', 'z']})

''' try:
    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())
    #print("Equalized DataFrames:")
    #print(df1_equal)
    #print(df2_equal)
except ValueError as e:
    print(e) '''


' try:\n    df1_equal, df2_equal = equalize_dataframe_sizes(df1.copy(), df2.copy())\n    #print("Equalized DataFrames:")\n    #print(df1_equal)\n    #print(df2_equal)\nexcept ValueError as e:\n    print(e) '

In [5]:
def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Create groups based on unique combinations of X and Y values
    groups_df1 = df1.groupby(['pessoa_gyro','atividade_gyro'])
    groups_df2 = df2.groupby(['pessoa', 'atividade'])

    # Iterate over groups from both DataFrames simultaneously
    for (x, y), group1 in groups_df1:
        #print(f"*******************IMPRIMINDO GRUPO1:{group1.shape}  --- {group1}")
        #if (x,y) in [(1637, 'C'),(1638, 'H'),(1638, 'O'),(1640, 'C'),(1640, 'E')]:
        #    continue
        #else:
        group2 = groups_df2.get_group((x, y))
        #print(f"*******************IMPRIMINDO GRUPO2: {group2.shape} - --- {group2}")        
        """  group1_has_nan = group1.isnull().sum().sum() > 0
            group2_has_nan = group2.isnull().sum().sum() > 0
            if group1_has_nan == True:
                    print(group1.isnull().sum().sum())
                    print (f"grupo {x,y} nulo")
                    break
            if group2_has_nan == True:
                    print(group2.isnull().sum().sum())
                    print (f"grupo {x,y} nulo")
                    break """
        
        if group2 is not None:  # Check if corresponding group exists in df2
            if group1.shape[0] != group2.shape[0]:
                group1, group2 = equalize_dataframe_sizes(group1.copy(), group2.copy())             
            #print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$Imprimindo dados grupo 1 :{group1}" )
            #print(f"$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$$Imprimindo shape1 :{group1.shape}" )
            #print("---------------------------------------------")
            #print(f"####################################Imprimindo dados grupo 2 :{group2}")
            #print(f"####################################Imprimindo dados grupo 2 shape :{group2.shape}")
            group1 = group1.reset_index(drop=True) 
            group2 = group2.reset_index(drop=True)  
            df_concat_group1_group2 = pd.concat([group1, group2], axis=1)
            df_concat_group1_group2 = df_concat_group1_group2.dropna()            
            has_nan = df_concat_group1_group2.isnull().sum().sum() > 0
            if has_nan == True:
                print (f"**************************** Imprimindo concat co nan: {df_concat_group1_group2}")
                break
            df_concat_group1_group2 = df_concat_group1_group2.reset_index(drop=True) 
            result_df = result_df.reset_index(drop=True)
            result_df = pd.concat([result_df, df_concat_group1_group2])

    return result_df

In [6]:
''' def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Iterate over consecutive groups in each DataFrame
    for x_val in df1['pessoa'].unique():
        group_df1 = df1[df1['pessoa'] == x_val]
        for y_val in group_df1['atividade'].unique():
            group1 = group_df1[group_df1['atividade'] == y_val]
            group2 = df2[(df2['pessoa'] == x_val) & (df2['atividade'] == y_val)]
            if not group2.empty:  # Check if corresponding group exists in df2
                result_df = pd.concat([result_df, pd.concat([group2,group1], keys=[1, 2], axis=1, ignore_index=True)])

    return result_df '''



' def custom_concat(df1, df2):\n    """\n    Concatenates two DataFrames with the same column names, considering specific conditions.\n\n    Args:\n        df1 (pd.DataFrame): The first DataFrame.\n        df2 (pd.DataFrame): The second DataFrame.\n\n    Returns:\n        pd.DataFrame: The concatenated DataFrame.\n    """\n\n    result_df = pd.DataFrame()\n\n    # Iterate over consecutive groups in each DataFrame\n    for x_val in df1[\'pessoa\'].unique():\n        group_df1 = df1[df1[\'pessoa\'] == x_val]\n        for y_val in group_df1[\'atividade\'].unique():\n            group1 = group_df1[group_df1[\'atividade\'] == y_val]\n            group2 = df2[(df2[\'pessoa\'] == x_val) & (df2[\'atividade\'] == y_val)]\n            if not group2.empty:  # Check if corresponding group exists in df2\n                result_df = pd.concat([result_df, pd.concat([group2,group1], keys=[1, 2], axis=1, ignore_index=True)])\n\n    return result_df '

In [7]:
''' # Sample DataFrames
df1 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'C', 'A', 'A', 'B', 'B']})
df2 = pd.DataFrame({'X': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],
                   'Y': ['A', 'A', 'B', 'B', 'A', 'B', 'B', 'B', 'B']})

def custom_concat(df1, df2):
    """
    Concatenates two DataFrames with the same column names, considering specific conditions.

    Args:
        df1 (pd.DataFrame): The first DataFrame.
        df2 (pd.DataFrame): The second DataFrame.

    Returns:
        pd.DataFrame: The concatenated DataFrame.
    """

    result_df = pd.DataFrame()

    # Iterate over consecutive groups in each DataFrame
    for x_val in df1['X'].unique():
        group_df1 = df1[df1['X'] == x_val]
        for y_val in group_df1['Y'].unique():
            group1 = group_df1[group_df1['Y'] == y_val]
            group2 = df2[(df2['X'] == x_val) & (df2['Y'] == y_val)]
            if not group2.empty:  # Check if corresponding group exists in df2
                result_df = pd.concat([result_df, pd.concat([group1, group2], keys=[1, 2], axis=1, ignore_index=False)])

    return result_df

# Concatenate the DataFrames
concatenated_df = custom_concat(df1.copy(), df2.copy()) 

print(concatenated_df) '''

' # Sample DataFrames\ndf1 = pd.DataFrame({\'X\': [1600, 1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601],\n                   \'Y\': [\'A\', \'A\', \'B\', \'B\', \'C\', \'A\', \'A\', \'B\', \'B\']})\ndf2 = pd.DataFrame({\'X\': [1600, 1600, 1600, 1600, 1601, 1601, 1601, 1601, 1601],\n                   \'Y\': [\'A\', \'A\', \'B\', \'B\', \'A\', \'B\', \'B\', \'B\', \'B\']})\n\ndef custom_concat(df1, df2):\n    """\n    Concatenates two DataFrames with the same column names, considering specific conditions.\n\n    Args:\n        df1 (pd.DataFrame): The first DataFrame.\n        df2 (pd.DataFrame): The second DataFrame.\n\n    Returns:\n        pd.DataFrame: The concatenated DataFrame.\n    """\n\n    result_df = pd.DataFrame()\n\n    # Iterate over consecutive groups in each DataFrame\n    for x_val in df1[\'X\'].unique():\n        group_df1 = df1[df1[\'X\'] == x_val]\n        for y_val in group_df1[\'Y\'].unique():\n            group1 = group_df1[group_df1[\'Y\'] == y_val]\n            

In [8]:
## get dataset

# select only sensor data from smart watches
csv_gyro_dir = WORKDIR +"Data/Raw/watch/gyro"
csv_accel_dir = WORKDIR +"Data/Raw/watch/accel"

In [9]:
df_csv_gyro = combine_csv_data(csv_gyro_dir)

In [10]:
df_csv_gyro = df_csv_gyro.replace({';':''}, regex=True)

In [11]:
df_csv_gyro.columns = ['pessoa_gyro','atividade_gyro','tempo_gyro','giroscópio_x','giroscópio_Y','giroscópio_Z']

In [12]:
df_csv_gyro.head()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,A,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,A,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,A,90426906196641,0.037975,0.254976,-0.1565635
4,1600,A,90426955696641,0.073129,0.719431,-0.0010349044


In [13]:
df_csv_gyro.tail()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3440337,1650,S,2426184371437252,-2.856065,2.057101,0.85393304
3440338,1650,S,2426184421424592,-2.911458,1.817417,1.2811041
3440339,1650,S,2426184471411932,-3.015854,1.550036,2.0576818
3440340,1650,S,2426184521399272,-3.218254,1.595842,3.0185504
3440341,1650,S,2426184571385262,-2.455525,0.866136,3.3541086


In [14]:
df_csv_gyro.shape

(3440342, 6)

In [15]:
df_csv_gyro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3440342 entries, 0 to 3440341
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_gyro     int64  
 1   atividade_gyro  object 
 2   tempo_gyro      int64  
 3   giroscópio_x    float64
 4   giroscópio_Y    float64
 5   giroscópio_Z    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 157.5+ MB


In [16]:
teste_gyro = df_csv_gyro[df_csv_gyro['pessoa_gyro']==1605]

In [17]:
teste_gyro.head()

,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
324935,1605,A,9914136743005,-1.067869,0.084463,-0.797466
324936,1605,A,9914186746890,-0.999692,-0.145634,-0.7260933
324937,1605,A,9914236752414,-0.604479,-0.177592,-0.6366111
324938,1605,A,9914286674776,-0.552281,-0.256422,-0.5098446
324939,1605,A,9914336690902,-0.495822,-0.227660,-0.4651035


In [18]:
teste_gyro.info()

<class 'pandas.core.frame.DataFrame'>
Index: 66187 entries, 324935 to 391121
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   pessoa_gyro     66187 non-null  int64  
 1   atividade_gyro  66187 non-null  object 
 2   tempo_gyro      66187 non-null  int64  
 3   giroscópio_x    66187 non-null  float64
 4   giroscópio_Y    66187 non-null  float64
 5   giroscópio_Z    66187 non-null  object 
dtypes: float64(2), int64(2), object(2)
memory usage: 3.5+ MB


In [19]:
''' condition1 = (df_csv_gyro['pessoa_gyro']== 1637) & (df_csv_gyro['atividade_gyro']=='C')
condition2 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='O')
condition3 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='C')
condition4 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='E')
condition5 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='H') '''

" condition1 = (df_csv_gyro['pessoa_gyro']== 1637) & (df_csv_gyro['atividade_gyro']=='C')\ncondition2 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='O')\ncondition3 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='C')\ncondition4 = (df_csv_gyro['pessoa_gyro']== 1640) & (df_csv_gyro['atividade_gyro']=='E')\ncondition5 = (df_csv_gyro['pessoa_gyro']== 1638) & (df_csv_gyro['atividade_gyro']=='H') "

In [20]:
#df_csv_gyro_filtered = df_csv_gyro[~(condition1 | condition2 | condition3 | condition4 | condition5)]

In [21]:
#df_csv_gyro_filtered = df_csv_gyro

In [22]:
#df_csv_gyro_filtered.info()

In [23]:
#test = df_csv_gyro[(df_csv_gyro['pessoa_gyro']==1638) & (df_csv_gyro['atividade_gyro']=='H') ]

In [24]:
#print(len(test))

In [25]:
#test2 = df_csv_gyro[df_csv_gyro['pessoa_gyro']==1638]

In [26]:
#print(len(test2))

In [27]:
df_csv_gyro.shape

(3440342, 6)

In [28]:
#df_csv_gyro.set_index(["pessoa_gyro","atividade_gyro","tempo_gyro"],inplace=True)

In [29]:
#df_csv_gyro.head()

In [30]:
df_csv_gyro.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3440342 entries, 0 to 3440341
Data columns (total 6 columns):
 #   Column          Dtype  
---  ------          -----  
 0   pessoa_gyro     int64  
 1   atividade_gyro  object 
 2   tempo_gyro      int64  
 3   giroscópio_x    float64
 4   giroscópio_Y    float64
 5   giroscópio_Z    object 
dtypes: float64(2), int64(2), object(2)
memory usage: 157.5+ MB


In [31]:
df_csv_accel = combine_csv_data(csv_accel_dir)

In [32]:
df_csv_accel = df_csv_accel.replace({';':''}, regex=True)

In [33]:
df_csv_accel.shape

(3777046, 6)

In [34]:
df_csv_accel.columns = ['pessoa','atividade','tempo_acelerômetro','acelerômetro_x','acelerômetro_Y','acelerômetro_Z']

In [35]:
df_csv_accel.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316
2,1600,A,90426807196641,3.253720,-0.191835,6.107758
3,1600,A,90426856696641,2.801216,-0.155922,5.997625
4,1600,A,90426906196641,3.770868,-1.051354,7.731027


In [36]:
df_csv_accel.tail()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z
3777041,1650,S,2426184421424592,1.486500,-13.302334,5.355231
3777042,1650,S,2426184471411932,4.017171,-13.146711,5.1182046
3777043,1650,S,2426184521399272,4.412214,-15.318253,3.3704374
3777044,1650,S,2426184571385262,4.426579,-13.012636,2.5755625
3777045,1650,S,2426184621371252,3.765780,-11.798776,2.166154


In [37]:
#df_csv_accel.set_index(["pessoa_acelerômetro","atividade_acelerômetro","tempo_acelerômetro"],inplace=True)

In [38]:
#df_csv_accel.head()

In [39]:
teste_accel = df_csv_accel[df_csv_accel['pessoa']==1605]

In [40]:
teste_accel.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z
325056,1605,A,9914136743005,6.753594,-4.764910,0.6823475
325057,1605,A,9914186746890,7.301866,-4.116082,0.7445967
325058,1605,A,9914236752414,7.610718,-3.840749,1.1133038
325059,1605,A,9914286674776,8.999355,-3.986795,1.0702082
325060,1605,A,9914336690902,10.996120,-3.754557,0.7206547


In [41]:
teste_accel.info()

<class 'pandas.core.frame.DataFrame'>
Index: 66195 entries, 325056 to 391250
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pessoa              66195 non-null  int64  
 1   atividade           66195 non-null  object 
 2   tempo_acelerômetro  66195 non-null  int64  
 3   acelerômetro_x      66195 non-null  float64
 4   acelerômetro_Y      66195 non-null  float64
 5   acelerômetro_Z      66195 non-null  object 
dtypes: float64(2), int64(2), object(2)
memory usage: 3.5+ MB


In [42]:
''' condition_accel1 = (df_csv_accel['pessoa']== 1637) & (df_csv_accel['atividade']=='C')
condition_accel2 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='O')
condition_accel3 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='C')
condition_accel4 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='E')
condition_accel5 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='H') '''

" condition_accel1 = (df_csv_accel['pessoa']== 1637) & (df_csv_accel['atividade']=='C')\ncondition_accel2 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='O')\ncondition_accel3 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='C')\ncondition_accel4 = (df_csv_accel['pessoa']== 1640) & (df_csv_accel['atividade']=='E')\ncondition_accel5 = (df_csv_accel['pessoa']== 1638) & (df_csv_accel['atividade']=='H') "

In [43]:
#df_csv_accel_filtered = df_csv_accel[~(condition_accel1 | condition_accel2 | condition_accel3 | condition_accel4 | condition_accel5)]

In [44]:
#df_csv_accel_filtered = df_csv_accel

In [45]:
#df_csv_accel_filtered.info()

In [46]:
#test3 = df_csv_accel_filtered[(df_csv_accel_filtered['pessoa']==1638) & (df_csv_accel_filtered['atividade']=='H') ]

In [47]:
#print(len(test3))

In [48]:
#test3.head()

In [49]:
#test4 = df_csv_accel_filtered[df_csv_accel_filtered['pessoa']==1638 ]

In [50]:
#print(len(test4))

In [51]:
df_csv_gyro.shape

(3440342, 6)

In [52]:
# Select rows with matching indices (inner join)
''' common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)
df1_filtered = df_csv_accel.loc[common_indices]
df2_filtered = df_csv_gyro.loc[common_indices]

# Concatenate horizontally
result = pd.concat([df1_filtered, df2_filtered], axis=1) '''

' common_indices = df_csv_accel.index.intersection(df_csv_gyro.index)\ndf1_filtered = df_csv_accel.loc[common_indices]\ndf2_filtered = df_csv_gyro.loc[common_indices]\n\n# Concatenate horizontally\nresult = pd.concat([df1_filtered, df2_filtered], axis=1) '

In [53]:
result_concatenation = custom_concat(df_csv_gyro.copy(),df_csv_accel.copy())

In [54]:
teste_result = result_concatenation[result_concatenation['pessoa']==1605]

In [55]:
teste_result.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
324933,1605,A,9914136743005,6.753594,-4.764910,0.6823475,1605,A,9914136743005,-1.067869,0.084463,-0.797466
324934,1605,A,9914186746890,7.301866,-4.116082,0.7445967,1605,A,9914186746890,-0.999692,-0.145634,-0.7260933
324935,1605,A,9914236752414,7.610718,-3.840749,1.1133038,1605,A,9914236752414,-0.604479,-0.177592,-0.6366111
324936,1605,A,9914286674776,8.999355,-3.986795,1.0702082,1605,A,9914286674776,-0.552281,-0.256422,-0.5098446
324937,1605,A,9914336690902,10.996120,-3.754557,0.7206547,1605,A,9914336690902,-0.495822,-0.227660,-0.4651035


In [56]:
result_concatenation.shape

(3440323, 12)

In [57]:
result_concatenation.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502,1600,A,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316,1600,A,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,A,90426807196641,3.253720,-0.191835,6.107758,1600,A,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,A,90426856696641,2.801216,-0.155922,5.997625,1600,A,90426906196641,0.037975,0.254976,-0.1565635
4,1600,A,90426906196641,3.770868,-1.051354,7.731027,1600,A,90426955696641,0.073129,0.719431,-0.0010349044


In [58]:
result_concatenation = result_concatenation.reset_index(drop=True)

In [59]:
result_concatenation.tail()


,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,pessoa_gyro,atividade_gyro,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3440318,1650,S,2426184321449912,4.768950,-5.401469,7.5842323,1650,S,2426184371437252,-2.856065,2.057101,0.85393304
3440319,1650,S,2426184371437252,2.137723,-7.211485,8.065467,1650,S,2426184421424592,-2.911458,1.817417,1.2811041
3440320,1650,S,2426184421424592,1.486500,-13.302334,5.355231,1650,S,2426184471411932,-3.015854,1.550036,2.0576818
3440321,1650,S,2426184471411932,4.017171,-13.146711,5.1182046,1650,S,2426184521399272,-3.218254,1.595842,3.0185504
3440322,1650,S,2426184521399272,4.412214,-15.318253,3.3704374,1650,S,2426184571385262,-2.455525,0.866136,3.3541086


In [60]:
test_diff_atividade = result_concatenation[result_concatenation['atividade']!=result_concatenation['atividade_gyro']]

In [61]:
print(test_diff_atividade.shape[0])

0


In [62]:
test_diff_pessoa = result_concatenation[result_concatenation['pessoa']!=result_concatenation['pessoa_gyro']]

In [63]:
print(len(test_diff_pessoa))

0


In [64]:
result_concatenation.columns

Index(['pessoa', 'atividade', 'tempo_acelerômetro', 'acelerômetro_x',
       'acelerômetro_Y', 'acelerômetro_Z', 'pessoa_gyro', 'atividade_gyro',
       'tempo_gyro', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z'],
      dtype='object')

In [65]:
result_concatenation = result_concatenation[['pessoa', 'atividade', 'tempo_acelerômetro', 'acelerômetro_x',
       'acelerômetro_Y', 'acelerômetro_Z', 'tempo_gyro', 'giroscópio_x', 'giroscópio_Y', 'giroscópio_Z']]

In [66]:
result_concatenation.head()

,pessoa,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,A,90426807196641,3.253720,-0.191835,6.107758,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,A,90426856696641,2.801216,-0.155922,5.997625,90426906196641,0.037975,0.254976,-0.1565635
4,1600,A,90426906196641,3.770868,-1.051354,7.731027,90426955696641,0.073129,0.719431,-0.0010349044


In [67]:
result_concatenation = result_concatenation.rename(columns={'pessoa':'pessoa_x'})

In [68]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,A,90426708196641,7.091625,-0.591667,8.195502,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,A,90426757696641,4.972757,-0.158317,6.6967316,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,A,90426807196641,3.253720,-0.191835,6.107758,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,A,90426856696641,2.801216,-0.155922,5.997625,90426906196641,0.037975,0.254976,-0.1565635
4,1600,A,90426906196641,3.770868,-1.051354,7.731027,90426955696641,0.073129,0.719431,-0.0010349044


In [69]:
# Define the mapping dictionary
mapping_dict = {'A':0,'B':1,'C':2,'D':3,'E':4,'F':5,'G':6,'H':7,'I':8,'J':9,'K':10,'L':11,'M':12,'O':13,'P':14,'Q':15,'R':16,'S':17}

In [70]:
result_concatenation["atividade"] = result_concatenation["atividade"].replace(to_replace=mapping_dict)

In [71]:
result_concatenation.head()

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
0,1600,0,90426708196641,7.091625,-0.591667,8.195502,90426757696641,0.314944,-1.022277,-0.3099616
1,1600,0,90426757696641,4.972757,-0.158317,6.6967316,90426807196641,0.387382,-0.618541,-0.048971802
2,1600,0,90426807196641,3.253720,-0.191835,6.107758,90426856696641,0.070999,-0.209480,-0.1959783
3,1600,0,90426856696641,2.801216,-0.155922,5.997625,90426906196641,0.037975,0.254976,-0.1565635
4,1600,0,90426906196641,3.770868,-1.051354,7.731027,90426955696641,0.073129,0.719431,-0.0010349044


In [72]:
result_concatenation.tail(50)

,pessoa_x,atividade,tempo_acelerômetro,acelerômetro_x,acelerômetro_Y,acelerômetro_Z,tempo_gyro,giroscópio_x,giroscópio_Y,giroscópio_Z
3440273,1650,17,2426182071909922,2.681207,-4.802918,8.158841,2426182121897632,-1.969764,1.562819,-0.003604855
3440274,1650,17,2426182121897632,0.720355,-4.225915,12.947245,2426182171885342,1.064109,1.739653,0.3596503
3440275,1650,17,2426182171885342,0.267851,-10.120441,10.83077,2426182221873052,4.321687,2.470424,-0.7982921
3440276,1650,17,2426182221873052,-1.841440,3.921553,7.802105,2426182271860762,1.949343,2.141257,-1.4576908
3440277,1650,17,2426182271860762,-3.955520,6.049998,6.001665,2426182321848472,1.746943,2.089059,-1.4459729
3440278,1650,17,2426182321848472,-8.684069,-0.682497,5.168483,2426182371836182,1.905668,1.122865,-0.97725654
3440279,1650,17,2426182371836182,-7.836522,0.833033,4.8404775,2426182421823892,-0.828866,0.741500,0.061376274
3440280,1650,17,2426182421823892,-9.036016,-0.972195,4.06715,2426182471816332,0.509106,0.121516,-0.069651246
3440281,1650,17,2426182471816332,-8.243536,-0.318578,6.3033347,2426182521808772,0.791401,-0.638018,-0.51280123
3440282,1650,17,2426182521808772,-6.045659,2.346168,6.892308,2426182571801212,0.905384,-1.036426,-0.51812756


In [73]:
result_concatenation.to_csv('activity_recognition_data_watch_concat_by_comparacao_grupos.csv', index=False)